In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
import re, html
import torch
import os

# Load the 5 files

In [ ]:
file1 = pd.read_csv("personas_metaads_Gemini_3.1_pro.csv")
file2 = pd.read_csv("personas_metaads_GPT_5.csv")
file3 = pd.read_csv("personas_metaads_Claude_Opus_4.6.csv")
file4 = pd.read_csv("personas_metaads_Grok_4.1_fast_reasoning.csv")
file5 = pd.read_csv("personas_metaads_DeepSeek_chat.csv")

# Drop "persona_cluster" and "cluster_rows" columns

In [ ]:
cols_to_drop = ["persona_cluster", "cluster_rows"]
file1 = file1.drop(columns=cols_to_drop, errors="ignore")
file2 = file2.drop(columns=cols_to_drop, errors="ignore")
file3 = file3.drop(columns=cols_to_drop, errors="ignore")
file4 = file4.drop(columns=cols_to_drop, errors="ignore")
file5 = file5.drop(columns=cols_to_drop, errors="ignore")

# Images path

In [ ]:
IMAGES_DIR = Path("images")         
OUT_DIR    = Path("data") / "reports"
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}

# Read images

In [ ]:
def read_images(images_dir: Path):
    files = [p for p in images_dir.rglob("*") if p.suffix.lower() in IMG_EXT]
    files.sort(key=lambda x: x.name.lower())
    return files

image_paths = read_images(IMAGES_DIR)
len(image_paths), image_paths[:3]

# Canonical column mapping

In [ ]:
def to_canonical(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Standardize expected fields
    if "persona_name" in df.columns and "Title" not in df.columns:
        df["Title"] = df["persona_name"].astype(str)
    if "persona_description" in df.columns and "Description" not in df.columns:
        df["Description"] = df["persona_description"].astype(str)
    if "persona_tactics" in df.columns and "Tactics" not in df.columns:
        df["Tactics"] = df["persona_tactics"].astype(str)

    # Meta ads (your naming)
    for i in (1,2,3):
        if f"ad_title_{i}" in df.columns:
            df[f"meta_title_{i}"] = df[f"ad_title_{i}"].astype(str)
        if f"ad_desc_{i}" in df.columns:
            df[f"meta_description_{i}"] = df[f"ad_desc_{i}"].astype(str)

    # Ensure columns exist even if missing
    needed = ["Title","Description","Tactics",
              "meta_title_1","meta_description_1",
              "meta_title_2","meta_description_2",
              "meta_title_3","meta_description_3"]
    for c in needed:
        if c not in df.columns:
            df[c] = ""

    return df

file1 = to_canonical(file1)
file2 = to_canonical(file2)
file3 = to_canonical(file3)
file4 = to_canonical(file4)
file5 = to_canonical(file5)

file1.columns

# Build a weighted text “profile” per persona

In [ ]:
JUNK = {"nan", "none", "null", "n/a"}

def safe_str(x) -> str:
    if x is None:
        return ""
    # pandas / numpy null handling
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    s = str(x).strip()
    return "" if s.lower() in JUNK else s

def normalize_text(t: str) -> str:
    t = safe_str(t)
    if not t:
        return ""
    t = html.unescape(t)
    # drop urls/emails
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)
    t = re.sub(r"\b\S+@\S+\.\S+\b", " ", t)
    # normalize separators/bullets
    t = re.sub(r"[•|/]+", ", ", t)
    # collapse whitespace
    t = re.sub(r"\s+", " ", t).strip()
    return t

# Stronger, more visual prompts
PROMPTS_EN = [
    "Lifestyle hotel marketing photo showing {t}.",
    "Professional hospitality photography of {t}, relaxing vacation mood.",
    "Travel scene featuring {t}, high-end getaway, natural light.",
    "Hotel interior or exterior photo depicting {t}."
]

def extract_profile_text(row: pd.Series) -> dict:
    title = normalize_text(row.get("Title"))
    desc  = normalize_text(row.get("Description"))
    tact  = normalize_text(row.get("Tactics"))
    meta  = normalize_text(" ".join([
        safe_str(row.get("meta_title_1")), safe_str(row.get("meta_description_1")),
        safe_str(row.get("meta_title_2")), safe_str(row.get("meta_description_2")),
        safe_str(row.get("meta_title_3")), safe_str(row.get("meta_description_3")),
    ]))

    return {"title": title, "desc": desc, "tact": tact, "meta": meta}

def l2_normalize(v: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(v)
    return v if n < eps else (v / n)

def shorten(t: str, max_words: int = 60) -> str:
    words = t.split()
    return " ".join(words[:max_words])

def build_weighted_texts(profile: dict) -> list[tuple[str, float]]:
    parts = []
    for k, w in [("title",0.45), ("desc",0.35), ("tact",0.15), ("meta",0.05)]:
        txt = profile.get(k, "")
        if not txt:
            continue

        variants = [p.format(t=txt) for p in PROMPTS_EN]
        w_each = w / len(variants)   # ✅ split weight across variants

        for v in variants:
            parts.append((v, w_each))

    return parts



# Retrieval backbone (CLIP) for text↔image matching

In [ ]:
import torch
import open_clip
from PIL import Image, ImageOps
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-16",
    pretrained="laion2b_s34b_b88k",
    device=DEVICE
)
clip_tokenizer = open_clip.get_tokenizer("ViT-B-16")
clip_model.eval()

@torch.inference_mode()
def clip_text_embed(texts, batch_size=64):
    outs = []
    for i in range(0, len(texts), batch_size):
        tokens = clip_tokenizer(texts[i:i+batch_size]).to(DEVICE)
        feats = clip_model.encode_text(tokens)
        feats = torch.nn.functional.normalize(feats, dim=1)
        outs.append(feats.cpu())
    return torch.cat(outs).numpy()

@torch.inference_mode()
def clip_image_embed(images, batch_size=16):
    outs = []
    for i in range(0, len(images), batch_size):
        batch = []
        for im in images[i:i+batch_size]:
            im = ImageOps.exif_transpose(im).convert("RGB")
            batch.append(clip_preprocess(im))
        batch = torch.stack(batch).to(DEVICE)
        feats = clip_model.encode_image(batch)
        feats = torch.nn.functional.normalize(feats, dim=1)
        outs.append(feats.cpu())
    return torch.cat(outs).numpy()

# Embed all images once

In [ ]:
def embed_all_images(paths: list[Path], batch_size: int = 16):
    vecs = []
    valid = []
    failed = 0
    emb_dim = None

    for i in tqdm(range(0, len(paths), batch_size), desc="Embedding images"):
        batch_paths = paths[i:i + batch_size]
        imgs = []
        keep = []

        for p in batch_paths:
            try:
                with Image.open(p) as im:
                    im = ImageOps.exif_transpose(im)
                    if im.mode != "RGB":
                        im = im.convert("RGB")
                    imgs.append(im.copy())
                    keep.append(p)
            except Exception:
                failed += 1
                continue

        if not imgs:
            continue

        embs = clip_image_embed(imgs)

        if emb_dim is None:
            emb_dim = embs.shape[1]

        vecs.append(embs)
        valid.extend(keep)

        # free memory explicitly
        del imgs, embs

    if vecs:
        mat = np.vstack(vecs)
    else:
        mat = np.zeros((0, emb_dim or 0), dtype=np.float32)

    if failed > 0:
        print(f"[embed_all_images] Skipped {failed} unreadable images")

    return valid, mat


# Embed persona profiles + retrieve top-K images per persona

In [ ]:
# open_clip CLIP models don't have .config, so infer TEXT_DIM from encode_text()

with torch.inference_mode():
    _dummy = clip_tokenizer(["test"]).to(DEVICE)
    TEXT_DIM = int(clip_model.encode_text(_dummy).shape[-1])  # e.g., 512 or 768 depending on model


def embed_persona_row(row: pd.Series) -> np.ndarray:
    profile = extract_profile_text(row)
    parts = build_weighted_texts(profile)  # list[(text, weight)]

    if not parts:
        return np.zeros((TEXT_DIM,), dtype=np.float32)

    # deduplicate while preserving weights by summing
    w_map = {}
    for t, w in parts:
        if not t:
            continue
        w_map[t] = w_map.get(t, 0.0) + float(w)

    texts = list(w_map.keys())
    weights = np.array([w_map[t] for t in texts], dtype=np.float32)

    # normalize weights
    s = weights.sum()
    if s > 0:
        weights /= s

    embs = clip_text_embed(texts)  # (n, d), already normalized row-wise
    weighted = (embs * weights[:, None]).sum(axis=0)
    return l2_normalize(weighted.astype(np.float32))


def mmr_rerank(query_vec: np.ndarray, doc_mat: np.ndarray, topn: int = 10, lambda_rel: float = 0.7):
    # assumes vectors are L2-normalized
    sim_q = doc_mat @ query_vec

    selected = []
    candidates = list(range(doc_mat.shape[0]))

    first = int(np.argmax(sim_q))
    selected.append(first)
    candidates.remove(first)

    while len(selected) < topn and candidates:
        sel_mat = doc_mat[selected]
        best, best_score = None, -1e9

        for c in candidates:
            rel = sim_q[c]
            div = np.max(sel_mat @ doc_mat[c])  # similarity to already selected
            score = lambda_rel * rel - (1 - lambda_rel) * div
            if score > best_score:
                best_score = score
                best = c

        selected.append(best)
        candidates.remove(best)

    return selected


def retrieve_topk_unique_mmr(df: pd.DataFrame, k: int = 10, lambda_rel: float = 0.7):
    persona_vecs = []
    for idx in tqdm(range(len(df)), desc="Embedding personas"):
        persona_vecs.append(embed_persona_row(df.iloc[idx]))
    persona_mat = np.vstack(persona_vecs).astype(np.float32)

    # if persona_mat and image_mat are L2-normalized, cosine similarity == dot product
    # (this avoids sklearn cosine_similarity overhead)
    sims = persona_mat @ image_mat.T

    used = set()
    assigned = {}

    for i in range(len(df)):
        q = persona_mat[i]

        # only consider unused images first
        available_idx = [j for j in range(image_mat.shape[0]) if j not in used]
        if len(available_idx) == 0:
            available_idx = list(range(image_mat.shape[0]))  # fallback

        sub_mat = image_mat[available_idx]
        chosen_local = mmr_rerank(q, sub_mat, topn=min(k, len(available_idx)), lambda_rel=lambda_rel)
        chosen_global = [available_idx[c] for c in chosen_local]

        # mark used
        for j in chosen_global:
            used.add(j)

        # if fewer than k, fill with best remaining by similarity
        if len(chosen_global) < k:
            order = np.argsort(-sims[i])
            for j in order:
                j = int(j)
                if j in chosen_global:
                    continue
                chosen_global.append(j)
                if len(chosen_global) >= k:
                    break

        assigned[i] = [valid_image_paths[j] for j in chosen_global[:k]]

    return assigned, sims

# English captions

In [ ]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration

CAPTION_MODEL = "Salesforce/blip2-opt-2.7b"

blip2_processor = Blip2Processor.from_pretrained(CAPTION_MODEL)

blip2_model = Blip2ForConditionalGeneration.from_pretrained(
    CAPTION_MODEL,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto"
)
blip2_model.eval()

@torch.inference_mode()
def caption_image_en(im, max_new_tokens=50):
    im = ImageOps.exif_transpose(im).convert("RGB")
    inputs = blip2_processor(
        images=im,
        return_tensors="pt"
    ).to(DEVICE, torch.float16 if DEVICE=="cuda" else torch.float32)

    out = blip2_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=5,
        do_sample=False
    )
    return blip2_processor.decode(out[0], skip_special_tokens=True).strip()


# stronger stopwords + phrase extraction (bigrams)
STOP = {
    "this","that","with","from","your","they","them","have","will","into","more","than",
    "stay","hotel","guest","guests","offer","offers","booking","bookings","reservation",
    "experience","perfect","enjoy","great","best","time","make","discover","welcome",
    "includes","including","available","today","night","nights","day","days"
}

def extract_keywords(text: str, n: int = 8):
    text = normalize_text(text)
    words = re.findall(r"[a-z]{3,}", text)
    words = [w for w in words if w not in STOP]

    freq = {}
    # unigrams
    for w in words:
        freq[w] = freq.get(w, 0) + 1
    # bigrams (phrases) boosted
    for i in range(len(words) - 1):
        bg = f"{words[i]} {words[i+1]}"
        freq[bg] = freq.get(bg, 0) + 2

    top = sorted(freq.items(), key=lambda x: (-x[1], x[0]))[:n]
    return [k for k,_ in top]


def build_rationale(persona_row: pd.Series, caption: str) -> str:
    persona_text = " ".join([
        str(persona_row.get("Title","") or ""),
        str(persona_row.get("Description","") or ""),
        str(persona_row.get("Tactics","") or "")
    ])

    kws = extract_keywords(persona_text, n=10)
    cap = normalize_text(caption)
    cap_tokens = set(re.findall(r"[a-z]{3,}", cap))

    # keep only keywords that overlap with caption tokens
    def overlaps(kw: str) -> bool:
        return any(p in cap_tokens for p in kw.split())

    matched = [kw for kw in kws if overlaps(kw)][:6]

    if not caption:
        return "Selected because the image matches the persona’s travel/hospitality context."

    if matched:
        return (
            f"Selected because the image shows {caption.lower()}, "
            #f"aligning with persona themes such as: {', '.join(matched)}."
        )

    # fallback (still grounded)
    if kws:
        return (
            f"Selected because the image shows {caption.lower()} and fits the persona’s broader interests "
            #f"(e.g., {', '.join(kws[:4])})."
        )

    return f"Selected because the image shows {caption.lower()} and fits the general travel/hospitality context."


# HTML report generator

In [ ]:
REPORT_CSS = """
:root{--bg:#f7f4ee;--fg:#111;--muted:#555;--card:#fff;--border:#ded5c5;--accent:#f0b400;}
*{box-sizing:border-box;}
body{background:var(--bg);color:var(--fg);font:16px/1.5 system-ui,-apple-system,Segoe UI,Roboto,Arial,sans-serif;margin:24px;}
h1{margin:0 0 14px;}
.section{border:1px solid var(--border);background:var(--card);border-radius:14px;padding:16px;margin-bottom:18px;}
.meta{display:flex;gap:8px;flex-wrap:wrap;margin-bottom:8px;}
.chip{background:#f0e7d6;padding:4px 8px;border-radius:999px;font-size:12px;color:var(--muted);}
.grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(240px,1fr));gap:12px;margin-top:12px;}
.figure{border:1px solid var(--border);border-radius:12px;padding:8px;background:var(--card);}
.figure img{width:100%;height:180px;object-fit:cover;border-radius:8px;border:1px solid var(--border);}
.filename{font-size:12px;color:var(--muted);margin-top:6px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;}
.caption{font-size:14px;margin-top:6px;}
.reason{font-size:13px;color:var(--muted);margin-top:4px;}
details{border:1px dashed var(--border);border-radius:10px;padding:8px 10px;margin-top:8px;}
summary{cursor:pointer;font-weight:600;}
"""

def build_html_report(df: pd.DataFrame, assigned: dict[int, list[Path]], out_html: Path, report_title: str, max_images: int = 10):
    out_html.parent.mkdir(parents=True, exist_ok=True)

    parts = [f"""<!doctype html><html><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>{html.escape(report_title)}</title>
<style>{REPORT_CSS}</style></head><body>
<h1>{html.escape(report_title)}</h1>
"""]

    for ridx in range(len(df)):
        row = df.iloc[ridx]
        title = str(row.get("Title","")).strip()
        desc = str(row.get("Description","")).strip()
        tact = str(row.get("Tactics","")).strip()

        m = []
        for i in (1,2,3):
            m.append((str(row.get(f"meta_title_{i}","")).strip(), str(row.get(f"meta_description_{i}","")).strip()))

        section = [f"""
<div class="section">
  <div class="meta">
    <span class="chip">Persona #{ridx}</span>
    <span class="chip">{min(max_images, len(assigned.get(ridx, [])))} images</span>
  </div>
  <h2>{html.escape(title)}</h2>
  <h3>Description</h3><p>{html.escape(desc)}</p>
  <h3>Tactics</h3><p>{html.escape(tact)}</p>
  <details><summary>Meta Ads</summary>
    <p><b>Ad 1:</b> {html.escape(m[0][0])} — {html.escape(m[0][1])}</p>
    <p><b>Ad 2:</b> {html.escape(m[1][0])} — {html.escape(m[1][1])}</p>
    <p><b>Ad 3:</b> {html.escape(m[2][0])} — {html.escape(m[2][1])}</p>
  </details>
  <div class="grid">
"""]

        imgs = assigned.get(ridx, [])[:max_images]
        for p in imgs:
            try:
                im = Image.open(p).convert("RGB")
                cap = caption_image_en(im)
                why = build_rationale(row, cap)
            except Exception:
                cap, why = "", "Unable to caption this image."

            # Use relative path so it works when you open HTML locally
            rel = p.as_posix()
            section.append(f"""
    <div class="figure">
      <img src="{html.escape(rel)}" alt="{html.escape(p.name)}">
      <div class="filename">{html.escape(p.name)}</div>
      <div class="caption">{html.escape(cap)}</div>
      <div class="reason">{html.escape(why)}</div>
    </div>
""")

        section.append("  </div>\n</div>\n")
        parts.append("".join(section))

    parts.append("</body></html>")
    out_html.write_text("".join(parts), encoding="utf-8")
    return str(out_html)


In [ ]:
try:
    image_mat
    valid_image_paths
except NameError:
    valid_image_paths, image_mat = embed_all_images(image_paths, batch_size=16)

print(image_mat.shape, len(valid_image_paths))

# Run report generation for all 5 model outputs

In [ ]:
reports = {}

assigned1, _ = retrieve_topk_unique_mmr(file1, k=10, lambda_rel=0.7)
reports["Gemini_3.1_pro"] = build_html_report(
    file1, assigned1,
    OUT_DIR / "visual_report_Gemini_3.1_pro.html",
    report_title="Visual Persona Report — Gemini 3.1 Pro",
    max_images=10
)

assigned2, _ = retrieve_topk_unique_mmr(file1, k=10, lambda_rel=0.7)
reports["GPT_5"] = build_html_report(
    file2, assigned2,
    OUT_DIR / "visual_report_GPT_5.html",
    report_title="Visual Persona Report — GPT-5",
    max_images=10
)

assigned3, _ = retrieve_topk_unique_mmr(file1, k=10, lambda_rel=0.7)
reports["Claude_Opus_4.6"] = build_html_report(
    file3, assigned3,
    OUT_DIR / "visual_report_Claude_Opus_4.6.html",
    report_title="Visual Persona Report — Claude Opus 4.6",
    max_images=10
)

assigned4, _ = retrieve_topk_unique_mmr(file1, k=10, lambda_rel=0.7)
reports["Grok_4.1_fast_reasoning"] = build_html_report(
    file4, assigned4,
    OUT_DIR / "visual_report_Grok_4.1_fast_reasoning.html",
    report_title="Visual Persona Report — Grok 4.1 Fast Reasoning",
    max_images=10
)

assigned5, _ = retrieve_topk_unique_mmr(file1, k=10, lambda_rel=0.7)
reports["DeepSeek_chat"] = build_html_report(
    file5, assigned5,
    OUT_DIR / "visual_report_DeepSeek_chat.html",
    report_title="Visual Persona Report — DeepSeek Chat",
    max_images=10
)

reports
